In [3]:
import tqdm
import json

import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity

## User-based Collaborative Filtering

#### Основная идея: 
Рекомендовать пользователю треки, которые понравились похожим на него пользователям

$$\hat r_{ui} = h^{-1} \left( \frac{\sum_{v \in N_i(u)} w_{uv} h(r_{vi})}{\sum_{v \in N_i(u)} w_{uv}} \right)$$

$N_i(u)$ - соседи пользователя $u$, которые оценили айтем $i$,
$w_{uv}, w_{ij}$ - веса соседей, 
$h$ - функция нормализации



**Нормализация**: В качестве функции нормализации используем среднее время прослушивания

**Веса**: Похожих пользователей будем искать по *cosine similarity*

**Отсутствующие данные**: заполним средним времнем прослушивания по пользователю

**Соседи**: в качестве соседей будем рассматривать всех пользователей. Q: Как это упростит формулу?

In [6]:
BOTIFY_DATA_DIR = "/Users/anndosova/RecSys_v2/dosova-made-recsys-course/botify/data/"

data = pd.read_json("/Users/anndosova/RecSys_v2/logs/experiments/seminar3/data.json", lines=True)[["user", "time", "track"]].copy()

data.head()

,user,time,track
0,9352,1.00,1114
1,1837,1.00,9665
2,1837,0.08,35349
3,9352,0.06,13367
4,1837,0.26,24358


In [8]:
data["normalized_time"] = data.groupby("user")["time"].transform(lambda time: time - time.mean())

data.head()

,user,time,track,normalized_time
0,9352,1.00,1114,0.726522
1,1837,1.00,9665,0.610000
2,1837,0.08,35349,-0.310000
3,9352,0.06,13367,-0.213478
4,1837,0.26,24358,-0.130000


In [9]:
interactions = pd.pivot_table(data, values="normalized_time", index="user", columns="track").fillna(0)

print(f"Interactions matrix: shape={interactions.shape}, sparsity={(interactions != 0).values.sum() / interactions.size}")

Interactions matrix: shape=(6324, 35638), sparsity=0.0003026071531929456


In [10]:
interactions.head()

track,0,1,2,3,4,5,7,9,10,11,...,49981,49982,49983,49984,49985,49986,49988,49989,49994,49998
user,,,,,,,,,,,,,,,,,,,,,
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [12]:
interactions.loc[0][interactions.loc[0] != 0]

track
491      0.831667
519     -0.168333
832     -0.168333
1289     0.831667
4296    -0.168333
18362    0.831667
18891   -0.168333
22383   -0.138333
22972   -0.168333
24267   -0.168333
28172   -0.168333
29260   -0.168333
29570   -0.168333
31896   -0.168333
34557   -0.168333
34627   -0.168333
35749   -0.168333
48612   -0.168333
Name: 0, dtype: float64

In [13]:
similarity_matrix = cosine_similarity(interactions)
np.fill_diagonal(similarity_matrix, 0)

print(f"Mean positive neighbours per user: {(similarity_matrix > 0).sum(axis=1).mean()}")

Mean positive neighbours per user: 13.860531309297913


In [14]:
print(f"Mean negative neighbours per user: {(similarity_matrix < 0).sum(axis=1).mean()}")

Mean negative neighbours per user: 5.8336495888678055


In [17]:
# TODO: Compute proper user-based scores
# TODO: expected size: observed users x observed tracks
scores_matrix = np.matmul(similarity_matrix, interactions.values)

scores = pd.DataFrame(
    scores_matrix,
    index=interactions.index,
    columns=interactions.columns
)

scores[[1, 2, 3, 4, 5]].head()

track,1,2,3,4,5
user,,,,,
0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0


In [18]:
(scores != 0).values.sum() / scores.size

0.007266627144930084

## Глянем на рекомендации

In [24]:
tracks = pd.read_json(BOTIFY_DATA_DIR + "tracks.json", lines=True).set_index("track")
tracks.head()

,artist,title,genre,pop
track,,,,
7,Harmonia,Sehr kosmisch,Pop_Rock,65688
0,Björk,Undo,None,57660
2,Dwight Yoakam,You're The One,Country,55035
1,Florence + The Machine,Dog Days Are Over (Radio Edit),None,52773
15,Kings Of Leon,Revelry,Pop_Rock,48290


In [20]:
user = np.random.choice(scores.index)
k = 10

# data[data["user"] == user]

In [21]:
data[data["user"] == user]

,user,time,track,normalized_time
68159,6054,1.00,84,0.81
68161,6054,0.00,24735,-0.19
68163,6054,0.14,9903,-0.05
68165,6054,0.00,44894,-0.19
68166,6054,0.00,28475,-0.19
68168,6054,0.00,7554,-0.19


In [25]:
user_scores = pd.merge(
    scores.loc[user].sort_values(ascending=False)[:k].to_frame("score"),
    tracks, 
    left_index=True, 
    right_index=True,
    how="inner"
)

user_scores

,score,artist,title,genre,pop
track,,,,,
84,8.105605,Taylor Swift,Love Story,Country,14836
542,0.485729,Taylor Swift,You Belong With Me,Country,10852
590,0.385852,Corinne Bailey Rae,Put Your Records On,None,2991
14587,0.383114,Train,Words,None,385
39166,0.354208,Atlas Sound,Cold As Ice,Pop_Rock,78
12310,0.354208,Taking Back Sunday,Spin (Album Version),Pop_Rock,382
5681,0.287164,Thousand Foot Krutch,Welcome To The Masquerade,Pop_Rock,579
15087,0.279003,Muse,Invincible,Pop_Rock,1498
85,0.279003,John Mayer,Heartbreak Warfare,Pop_Rock,14819


In [26]:
user_interactions = pd.merge(
    interactions.loc[user].sort_values(ascending=False).to_frame("time"),
    tracks, 
    left_index=True, 
    right_index=True, 
    how="inner"
)

user_interactions[user_interactions["time"] != 0]

,time,artist,title,genre,pop
track,,,,,
84,0.81,Taylor Swift,Love Story,Country,14836
9903,-0.05,Bright Eyes,Drunk Kid Catholic,None,1687
28475,-0.19,Born Of Osiris,Brace Legs (feat. NO),Pop_Rock,84
44894,-0.19,Egberto Gismonti,Cancao Do Carreiro,Latin,89
7554,-0.19,Death Cab for Cutie,The Ice Is Getting Thinner (Album Version),Pop_Rock,585
24735,-0.19,3 Inches Of Blood,Destroy The Orcs (Album Version),Pop_Rock,95


## Подготавливаем рекомендации для продакшена

In [27]:
def recommend(user_id, scores, k):
    return scores.loc[user_id].sort_values(ascending=False)[:k].index.tolist()

In [28]:
users = data["user"].unique()

with open(BOTIFY_DATA_DIR + "recommendations_ub.json", "w") as rf:
    for user in tqdm.tqdm(users):
        recommendation = {
            "user": int(user),
            "tracks": recommend(user, scores, 100)
        }
        rf.write(json.dumps(recommendation) + "\n")

100%|██████████████████████████████████████| 6324/6324 [00:10<00:00, 598.51it/s]
